## Playground for visualizing the dataset
CO: 07/16/2026 $\newline$
CB: SK $\newline$
UO: 

In [ ]:
#imports
import os, sys, glob, re
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import h5py
from scipy.signal import butter, sosfiltfilt
from scipy.ndimage import uniform_filter1d
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
DATA_ROOT    = '/Volumes/SanDisk/files/derivative'
OUTPUT_DIR   = os.path.expanduser('~/Desktop/NTS_decoding_results')
# STIM_DUR_S   = 5.0     # seconds of stim to extract per block
# PRE_STIM_S   = 5.0     # seconds before stim onset  (sham window)
# RMS_WIN_S    = 0.5     # RMS smoothing window
# N_CH         = 32

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ── Data loading ──────────────────────────────────────────────────────────────
def load_stim_file(sub_dir: str, stim_num: int):
    """
    Load LFP (noArti) and MUA (MUA_100 > mua_50 fallback) for one stim file.
    Returns dict with keys: lfp, mua, stim, time, fs  –– or None if missing.
    """
    base     = os.path.join(sub_dir, f'stim{stim_num}')
    lfp_path = base + '_noArti.mat'
    if not os.path.exists(lfp_path):
        return None

    with h5py.File(lfp_path, 'r') as f:
        lfp      = np.array(f['data']).squeeze()    # (T, 32)
        stim_sig = np.array(f['stim']).squeeze()    # (T,)
        time     = np.array(f['time']).squeeze()    # (T,)
        fs       = float(np.array(f['fs']).flat[0])

    # MUA: prefer explicit MUA file; fall back to spike_train inside LFP file
    mua = None
    for suffix in ('_MUA_100.mat', '_mua_50.mat', '_MUA_50.mat'):
        mua_path = base + suffix
        if os.path.exists(mua_path):
            with h5py.File(mua_path, 'r') as f:
                mua = np.array(f['spike_train'])    # (T', 32)
            break

    # Align lengths if MUA differs
    T = lfp.shape[0]
    if mua is not None and mua.shape[0] != T:
        n = min(mua.shape[0], T)
        lfp      = lfp[:n]
        stim_sig = stim_sig[:n]
        time     = time[:n]
        mua      = mua[:n]

    return dict(lfp=lfp, mua=mua, stim=stim_sig, time=time, fs=fs)

## Per-subject LFP overview (<300 Hz), all stim conditions overlaid

For each subject: load every `stim*_noArti.mat` file, average across the 32 channels,
low-pass filter to <300 Hz, downsample for plotting, and overlay all stim conditions
on one axis (labeled by stim number).

In [ ]:
LOWPASS_HZ  = 300.0    # LFP cutoff
PLOT_FS     = 1000.0   # target sample rate for plotting (well above 2x cutoff)
READ_CHUNK  = 500_000   # rows per HDF5 read, keeps memory bounded on huge files


def load_noArti_mean(sub_path: str, stim_num: int, chunk: int = READ_CHUNK):
    """Stream stim{N}_noArti.mat in row chunks, returning (mean-across-channels signal, fs)."""
    path = os.path.join(sub_path, f'stim{stim_num}_noArti.mat')
    with h5py.File(path, 'r') as f:
        dset = f['data']              # (T, 32)
        T    = dset.shape[0]
        fs   = float(np.array(f['fs']).flat[0])
        mean_sig = np.empty(T, dtype=np.float64)
        for start in range(0, T, chunk):
            end = min(start + chunk, T)
            mean_sig[start:end] = dset[start:end, :].mean(axis=1)
    return mean_sig, fs


def lowpass_filter(sig: np.ndarray, fs: float, cutoff: float = LOWPASS_HZ, order: int = 4):
    """Zero-phase Butterworth low-pass."""
    sos = butter(order, cutoff, btype='low', fs=fs, output='sos')
    return sosfiltfilt(sos, sig)


def decimate_for_plot(sig: np.ndarray, fs: float, target_fs: float = PLOT_FS):
    """Simple decimation for rendering only — signal is already band-limited below target_fs/2."""
    factor = max(1, int(round(fs / target_fs)))
    sig_ds = sig[::factor]
    t_ds   = np.arange(len(sig_ds)) * (factor / fs)
    return sig_ds, t_ds

In [ ]:
subjects = sorted(glob.glob(os.path.join(DATA_ROOT, 'sub-*')))

for sub_path in subjects:
    sub_name = os.path.basename(sub_path)

    files = glob.glob(os.path.join(sub_path, 'stim*_noArti.mat'))
    stim_nums = sorted(
        int(re.search(r'stim(\d+)_', os.path.basename(f)).group(1)) for f in files
    )
    print(f'{sub_name}: stim conditions {stim_nums}')

    fig, ax = plt.subplots(figsize=(16, 6))
    for sn in stim_nums:
        mean_sig, fs = load_noArti_mean(sub_path, sn)
        filt         = lowpass_filter(mean_sig, fs)
        filt_ds, t_ds = decimate_for_plot(filt, fs)
        ax.plot(t_ds, filt_ds, linewidth=0.8, alpha=0.8, label=f'stim {sn}')

    ax.set_title(f'{sub_name} — LFP (<{int(LOWPASS_HZ)} Hz), mean across channels')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('LFP amplitude (µV)')
    ax.legend(fontsize=8, ncol=6, loc='upper right')
    plt.tight_layout()

    out = os.path.join(OUTPUT_DIR, f'{sub_name}_noArti_lowpass_overlay.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  [saved] {out}')

## LFP vs. stimulation signal (one example session per subject)

For each subject, take one representative stim file (the lowest stim number available),
load the LFP (<300 Hz, mean across channels) alongside the raw stim channel, and overlay
them on a dual-axis plot — LFP amplitude on the left axis, stim signal (RMS envelope) on
the right axis, since the two have very different scales/units.

In [ ]:
STIM_ENV_WIN_S = 0.5   # RMS smoothing window for the stim-channel envelope


def load_noArti_and_stim(sub_path: str, stim_num: int, chunk: int = READ_CHUNK):
    """Like load_noArti_mean, but also returns the raw stim channel from the same file."""
    path = os.path.join(sub_path, f'stim{stim_num}_noArti.mat')
    with h5py.File(path, 'r') as f:
        dset = f['data']              # (T, 32)
        T    = dset.shape[0]
        fs   = float(np.array(f['fs']).flat[0])
        mean_sig = np.empty(T, dtype=np.float64)
        for start in range(0, T, chunk):
            end = min(start + chunk, T)
            mean_sig[start:end] = dset[start:end, :].mean(axis=1)
        stim_sig = np.array(f['stim']).squeeze().astype(np.float64)
    return mean_sig, stim_sig, fs


def stim_envelope(stim_sig: np.ndarray, fs: float, win_s: float = STIM_ENV_WIN_S):
    """RMS envelope of the raw stim channel, for visualizing ON/OFF blocks."""
    win = max(1, int(win_s * fs))
    return np.sqrt(uniform_filter1d(stim_sig ** 2, size=win))

In [ ]:
for sub_path in subjects:
    sub_name = os.path.basename(sub_path)

    files = glob.glob(os.path.join(sub_path, 'stim*_noArti.mat'))
    stim_nums = sorted(
        int(re.search(r'stim(\d+)_', os.path.basename(f)).group(1)) for f in files
    )
    example_sn = stim_nums[0]

    mean_sig, stim_sig, fs = load_noArti_and_stim(sub_path, example_sn)

    lfp_filt      = lowpass_filter(mean_sig, fs)
    lfp_ds, t_ds  = decimate_for_plot(lfp_filt, fs)
    env_ds, _     = decimate_for_plot(stim_envelope(stim_sig, fs), fs)

    fig, ax1 = plt.subplots(figsize=(16, 6))
    ax1.plot(t_ds, lfp_ds, color='steelblue', linewidth=0.8, label='LFP (<300 Hz)')
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('LFP amplitude (µV)', color='steelblue')
    ax1.tick_params(axis='y', labelcolor='steelblue')

    ax2 = ax1.twinx()
    ax2.plot(t_ds, env_ds, color='salmon', linewidth=0.8, alpha=0.8, label='Stim (RMS envelope)')
    ax2.set_ylabel('Stim signal (a.u.)', color='salmon')
    ax2.tick_params(axis='y', labelcolor='salmon')

    ax1.set_title(f'{sub_name} — stim {example_sn}: LFP vs. stimulation signal')
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)
    plt.tight_layout()

    out = os.path.join(OUTPUT_DIR, f'{sub_name}_stim{example_sn}_lfp_vs_stim.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'[saved] {out}')

## Per-trial stimulation parameters (pulse width & amplitude), all subjects, all stim files

For every subject and every `stim*_noArti.mat` file, detect individual stimulation trains
(blocks) using the same adaptive RMS-threshold algorithm as `decode_stim_sham.py`, then
detect individual pulses within each train to extract pulse width and amplitude. Only the
`stim` and `fs` datasets are read from each file — the 32-channel `data`/LFP array is not
needed and is expensive to load for this. Results are collected across all subjects/files,
printed in full, and saved to CSV in `OUTPUT_DIR`.

**Note:** raw `stim` amplitude units differ by subject — sub-1/sub-2 record in Volts
(~2-10), sub-3/4/5 record at ~1e6 scale (µV) — see `detect_stim_blocks`'s docstring /
`decode_stim_sham.py`. Values below are reported as-recorded, not normalized.

In [ ]:
RMS_WIN_S = 0.5   # matches decode_stim_sham.py


def detect_stim_blocks(stim_sig: np.ndarray, fs: float):
    """Port of decode_stim_sham.py's detect_stim_blocks. Returns (onsets_samp, offsets_samp)."""
    win = int(RMS_WIN_S * fs)
    rms = np.sqrt(uniform_filter1d(stim_sig.astype(np.float64) ** 2, size=win))
    p10, p90 = np.percentile(rms, 10), np.percentile(rms, 90)
    if p10 < 1e-3:
        threshold = max(0.1, p90 * 0.05)
    elif p90 / (p10 + 1e-9) > 3.0:
        threshold = np.sqrt(p10 * p90)
    else:
        threshold = p10 * 2.0
    binary = (rms > threshold).astype(int)
    d = np.diff(binary, prepend=0)
    ons, offs = np.where(d == 1)[0], np.where(d == -1)[0]
    n = min(len(ons), len(offs))
    return ons[:n], offs[:n]


def detect_pulses(seg: np.ndarray, fs: float, rel_thresh: float = 0.3, merge_gap: int = 2):
    """Detect individual pulses within one stim-block segment -> list of (width_us, amplitude)."""
    peak = np.max(np.abs(seg))
    if peak < 1e-9:
        return []
    idx = np.where(np.abs(seg) > rel_thresh * peak)[0]
    if len(idx) == 0:
        return []
    groups, start, prev = [], idx[0], idx[0]
    for i in idx[1:]:
        if i - prev > merge_gap + 1:
            groups.append((start, prev))
            start = i
        prev = i
    groups.append((start, prev))
    return [((e - s + 1) / fs * 1e6, np.max(np.abs(seg[s:e + 1]))) for s, e in groups]


def load_stim_only(sub_path: str, stim_num: int):
    """Read only 'stim' and 'fs' (never the 32-ch 'data' array) from a stim*_noArti.mat file."""
    path = os.path.join(sub_path, f'stim{stim_num}_noArti.mat')
    with h5py.File(path, 'r') as f:
        stim_sig = np.array(f['stim']).squeeze().astype(np.float64)
        fs       = float(np.array(f['fs']).flat[0])   # read per-file, never assumed constant
    return stim_sig, fs

In [ ]:
rows = []

for sub_path in subjects:                      # reuse existing `subjects = sorted(glob.glob(...))`
    sub_name = os.path.basename(sub_path)
    files = glob.glob(os.path.join(sub_path, 'stim*_noArti.mat'))
    stim_nums = sorted(
        int(re.search(r'stim(\d+)_', os.path.basename(f)).group(1)) for f in files
    )

    for sn in stim_nums:
        stim_sig, fs = load_stim_only(sub_path, sn)
        ons, offs = detect_stim_blocks(stim_sig, fs)

        for trial_idx, (on, off) in enumerate(zip(ons, offs)):
            seg = stim_sig[on:off]
            pulses = detect_pulses(seg, fs)
            duration_s = len(seg) / fs
            n_pulses = len(pulses)

            if n_pulses == 0:
                rows.append(dict(subject=sub_name, stim_file=f'stim{sn}', trial=trial_idx,
                                  onset_s=on / fs, duration_s=duration_s,
                                  mean_amp=np.nan, mean_width_us=np.nan,
                                  n_pulses=0, pulse_freq_hz=np.nan))
                continue

            widths = [p[0] for p in pulses]
            amps   = [p[1] for p in pulses]
            rows.append(dict(subject=sub_name, stim_file=f'stim{sn}', trial=trial_idx,
                              onset_s=on / fs, duration_s=duration_s,
                              mean_amp=np.mean(amps), mean_width_us=np.mean(widths),
                              n_pulses=n_pulses, pulse_freq_hz=n_pulses / duration_s))

    print(f'{sub_name}: processed stim files {stim_nums}')

trains_df = pd.DataFrame(rows)
pd.set_option('display.max_rows', None)
print(f'\nTotal trains detected: {len(trains_df)} across {len(subjects)} subjects\n')
print(trains_df.to_string(index=False))

out_csv = os.path.join(OUTPUT_DIR, 'stim_train_parameters_all_subjects.csv')
trains_df.to_csv(out_csv, index=False)
print(f'\n[saved] {out_csv}')
print('\nNOTE: mean_amp is raw/as-recorded and units differ by subject — sub-1/sub-2 store '
      'the stim monitor in Volts (~2-10), sub-3/4/5 in ~1e6-scale (µV); see detect_stim_blocks '
      'docstring / decode_stim_sham.py for the source of this discrepancy.')

## Cross-temporal decoding (SVM): Stim vs. Sham, per subject

For each subject, pool every stimulation block ("trial") across all its stim files, build
matched Stim/Sham epochs, and train an SVM at each time bin to decode Stim vs. Sham —
then test each trained model at every other time bin to build a **temporal generalization
matrix** (train-time × test-time decoding accuracy).

**Design choices** (data-driven, validated against the real dataset before implementing):
- **Trials kept**: only blocks with duration in [5.0, 5.6]s are treated as real stimulation
  trains (674/679 detected blocks qualify — this discards the ~5 truncated/odd-duration
  blocks, e.g. the ~1.4s block at the start of some files).
- **Epoch window**: `[onset - 5s, offset + 5s]` per trial (≈15.45s on average, since kept
  block durations cluster tightly around 5.45s) — a fixed number of bins (`n_bins`, at
  100ms resolution) is computed from the *median* kept-block duration, and every epoch is
  binned into that many equal-width chunks, so small duration variation across trials
  doesn't break alignment.
- **Sham epochs**: carved from the quiet gap between consecutive real blocks in the same
  file (regular gaps are consistently ~24.7–24.9s — far longer than the ~15.45s epoch
  needs), with a 2s buffer kept clear of the neighboring real blocks on each side. The
  **first gap of each file is excluded** — it sits right after the truncated/warm-up block
  and is a much longer (~22-106s), qualitatively different leading segment rather than a
  regular inter-trial interval, so it's left out to keep the sham pool homogeneous.
- **Features**: 32-channel LFP (`data`, low-pass <300 Hz), averaged within each 100ms bin
  — a (n_bins, 32) array per epoch.
- **Classifier**: same SVM config as `decode_stim_sham.py` (`StandardScaler` + `SVC(kernel='rbf',
  class_weight='balanced')`), evaluated with 5-fold stratified cross-validation per subject.
  Class counts don't need to be balanced — `class_weight='balanced'` and stratified CV
  handle that; what matters is having enough *absolute* samples per class per subject.

**Runtime note**: pipeline validated end-to-end on a real subset (extraction ≈0.8s/epoch
including the low-pass filter; TGM compute ≈15-60s/subject depending on trial count). With
~1300 epochs total across all subjects, expect this cell to take roughly **20-30 minutes**
end-to-end.

In [ ]:
DUR_MIN_S    = 5.0     # keep only blocks lasting ~5s ("stimulation trains")
DUR_MAX_S    = 5.6
GAP_MARGIN_S = 2.0      # buffer (s) kept clear of any real block when carving sham windows
PRE_S        = 5.0      # seconds before onset included in each epoch
POST_S       = 5.0      # seconds after offset included in each epoch
BIN_WIDTH_S  = 0.1       # temporal generalization matrix bin width

# ── Phase A: detect every block (cheap — 'stim' + 'fs' only), decide which are
#    "real" ~5s trains, and carve matched sham windows from the inter-block gaps ──
block_records = []
for sub_path in subjects:
    sub_name = os.path.basename(sub_path)
    files = glob.glob(os.path.join(sub_path, 'stim*_noArti.mat'))
    stim_nums = sorted(
        int(re.search(r'stim(\d+)_', os.path.basename(f)).group(1)) for f in files
    )
    for sn in stim_nums:
        stim_sig, fs = load_stim_only(sub_path, sn)
        ons, offs = detect_stim_blocks(stim_sig, fs)
        for on, off in zip(ons, offs):
            block_records.append(dict(subject=sub_name, sub_path=sub_path, stim_num=sn,
                                       fs=fs, onset_samp=int(on), offset_samp=int(off),
                                       duration_s=(off - on) / fs))

blocks_df = pd.DataFrame(block_records)
keep_mask = blocks_df['duration_s'].between(DUR_MIN_S, DUR_MAX_S)
ref_dur_s = blocks_df.loc[keep_mask, 'duration_s'].median()
n_bins    = int(round((PRE_S + ref_dur_s + POST_S) / BIN_WIDTH_S))

print(f'Kept {int(keep_mask.sum())}/{len(blocks_df)} blocks as ~5s stimulation trains '
      f'(duration in [{DUR_MIN_S}, {DUR_MAX_S}]s)')
print(f'Reference epoch length {PRE_S + ref_dur_s + POST_S:.2f}s -> '
      f'{n_bins} bins @ {int(BIN_WIDTH_S * 1000)}ms')

sham_records = []
need_len_s = PRE_S + ref_dur_s + POST_S
for (sub_path, sn), grp in blocks_df.groupby(['sub_path', 'stim_num']):
    grp = grp.sort_values('onset_samp')
    fs = grp['fs'].iloc[0]
    onsets  = grp['onset_samp'].to_numpy()
    offsets = grp['offset_samp'].to_numpy()
    margin   = int(GAP_MARGIN_S * fs)
    need_len = int(round(need_len_s * fs))
    # Skip i=0: the gap right after the file's first (often truncated/warm-up) block is
    # the long leading segment before real stimulation starts, not a regular inter-trial
    # recovery gap — excluding it keeps sham windows drawn from homogeneous ~24.7-24.9s
    # inter-block intervals only.
    for i in range(1, len(onsets) - 1):
        gap_start = offsets[i] + margin
        gap_end   = onsets[i + 1] - margin
        if gap_end - gap_start >= need_len:
            start = gap_start
            sham_records.append(dict(subject=grp['subject'].iloc[0], sub_path=sub_path,
                                      stim_num=sn, fs=fs, onset_samp=start,
                                      offset_samp=start + need_len))

sham_df = pd.DataFrame(sham_records)
print(f'Carved {len(sham_df)} matched sham (no-stim) windows from inter-block gaps '
      f"(excluding each file's leading gap)")

In [ ]:
def extract_epoch_features(sub_path: str, stim_num: int, start_samp: int, end_samp: int,
                            fs: float, n_bins: int):
    """Slice [start_samp:end_samp) directly from the LFP 'data' array, low-pass filter,
    and bin into n_bins equal chunks -> (n_bins, 32) feature array, or None if out of range."""
    path = os.path.join(sub_path, f'stim{stim_num}_noArti.mat')
    with h5py.File(path, 'r') as f:
        T = f['data'].shape[0]
        s, e = max(0, start_samp), min(T, end_samp)
        if e - s < n_bins:
            return None
        seg = f['data'][s:e, :].astype(np.float64)
    sos    = butter(4, LOWPASS_HZ, btype='low', fs=fs, output='sos')
    filt   = sosfiltfilt(sos, seg, axis=0)
    chunks = np.array_split(filt, n_bins, axis=0)
    return np.array([c.mean(axis=0) for c in chunks])   # (n_bins, 32)


def gather_subject_epochs(sub_name: str, n_bins: int):
    """Build (X, y) for one subject: X (n_trials, n_bins, 32), y (n_trials,) 1=stim / 0=sham."""
    X, y = [], []

    stim_rows = blocks_df[keep_mask & (blocks_df['subject'] == sub_name)]
    for _, r in stim_rows.iterrows():
        start = r['onset_samp'] - int(round(PRE_S * r['fs']))
        end   = r['offset_samp'] + int(round(POST_S * r['fs']))
        feat  = extract_epoch_features(r['sub_path'], r['stim_num'], start, end, r['fs'], n_bins)
        if feat is not None:
            X.append(feat); y.append(1)

    sham_rows = sham_df[sham_df['subject'] == sub_name]
    for _, r in sham_rows.iterrows():
        feat = extract_epoch_features(r['sub_path'], r['stim_num'],
                                       r['onset_samp'], r['offset_samp'], r['fs'], n_bins)
        if feat is not None:
            X.append(feat); y.append(0)

    return np.array(X), np.array(y)


def compute_tgm(X: np.ndarray, y: np.ndarray, n_splits: int = 5, random_state: int = 42):
    """Temporal generalization matrix: train an SVM per time bin, test across all time bins."""
    n_b = X.shape[1]
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    acc = np.zeros((n_b, n_b))
    for train_idx, test_idx in skf.split(np.zeros(len(y)), y):
        y_tr, y_te = y[train_idx], y[test_idx]
        for tb in range(n_b):
            clf = Pipeline([('sc', StandardScaler()),
                             ('svm', SVC(kernel='rbf', class_weight='balanced', random_state=42))])
            clf.fit(X[train_idx, tb, :], y_tr)
            for teb in range(n_b):
                acc[tb, teb] += (clf.predict(X[test_idx, teb, :]) == y_te).mean()
    return acc / n_splits

In [ ]:
t_axis = np.arange(n_bins) * BIN_WIDTH_S - PRE_S   # nominal bin start time (s) relative to onset

for sub_path in subjects:
    sub_name = os.path.basename(sub_path)
    X, y = gather_subject_epochs(sub_name, n_bins)
    n_stim, n_sham = int((y == 1).sum()), int((y == 0).sum())
    print(f'{sub_name}: {n_stim} stim epochs, {n_sham} sham epochs')

    if len(np.unique(y)) < 2 or min(n_stim, n_sham) < 5:
        print('  Insufficient data for both classes — skipping')
        continue

    tgm = compute_tgm(X, y)

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(tgm, origin='lower', aspect='auto', cmap='RdBu_r', vmin=0.3, vmax=0.9,
                   extent=[t_axis[0], t_axis[-1], t_axis[0], t_axis[-1]])
    ax.plot([t_axis[0], t_axis[-1]], [t_axis[0], t_axis[-1]], 'k--', linewidth=0.6)
    ax.axvline(0, color='k', linewidth=0.6, linestyle=':')
    ax.axhline(0, color='k', linewidth=0.6, linestyle=':')
    ax.set_xlabel('Testing time (s, relative to stim onset)')
    ax.set_ylabel('Training time (s, relative to stim onset)')
    ax.set_title(f'{sub_name} — Temporal Generalization (SVM): Stim vs. Sham')
    fig.colorbar(im, ax=ax, label='Decoding accuracy')
    plt.tight_layout()

    out = os.path.join(OUTPUT_DIR, f'{sub_name}_tgm_stim_vs_sham.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  [saved] {out}')